
# GPU V5 — Fixed 7×7 / 21×21 Specialized Shared-Memory NLM

## Mục tiêu

Notebook này đánh giá **GPU V5**, một nhánh tối ưu **kế thừa trực tiếp từ GPU V2**.

V5 **không thay đổi thuật toán NLM** và không loại bớt candidate như V4. Thay vào đó, V5 giữ nguyên dense NLM nhưng specialize kernel cho cấu hình benchmark cố định:

- Ảnh: `512×512`
- Patch: `7×7`
- Search window: `21×21`
- CUDA block: `16×16`
- `h = 0.12`

Các tối ưu chính của V5:

1. Giữ **Shared Memory tile + halo** từ V2.
2. Cố định toàn bộ geometry tại compile time.
3. Cache reference patch `7×7` một lần cho mỗi thread.
4. Cho compiler unroll phần tính patch distance `49` phần tử.
5. Vẫn xét đủ `21×21 = 441` candidate cho mỗi output pixel.

Do đó, về correctness, kỳ vọng:

\[
CPU \approx V2 \approx V3 \approx V5
\]

Trong khi về performance, câu hỏi chính là:

> **Fixed-size specialization của V5 có thể tiến gần hoặc vượt V3 Cross-Correlation trên workload 7×7 / 21×21 hay không?**



## 1. Thiết lập môi trường

Notebook tự tìm thư mục gốc của project bằng cách kiểm tra thư mục `src`.


In [ ]:

from pathlib import Path
import sys
import time
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cupy as cp

from skimage import io, color, transform
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("ROOT:", ROOT)
print("CuPy:", cp.__version__)
print("CUDA device:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())



## 2. Import các phiên bản NLM và benchmark utilities

- CPU: dense naive reference.
- V2: Shared Memory baseline mà V5 kế thừa.
- V3: Cross-Correlation accelerated NLM.
- V5: Fixed-configuration specialized Shared-Memory NLM.


In [ ]:

from nlm.cpu import nlm_cpu_naive
from nlm.gpu_v2 import nlm_gpu_v2
from nlm.gpu_v3 import nlm_gpu_v3
from nlm.gpu_v5 import nlm_gpu_v5, prepare_gpu_v5_kernel_launch

from nlm.benchmark import benchmark_cuda_kernel, benchmark_gpu_end_to_end

print("Imports OK")



## 3. Cấu hình benchmark cố định

Ta dùng đúng workload đã được chọn để thể hiện rõ chi phí của dense NLM:

\[
512^2 \text{ output pixels}
\]

Mỗi output pixel xét:

\[
21\times21=441 \text{ candidates}
\]

Mỗi candidate so sánh:

\[
7\times7=49 \text{ patch elements}
\]

Tổng số patch-element comparisons xấp xỉ:

\[
512^2 \times 441 \times 49 \approx 5.67\times10^9
\]

Đây là workload đủ lớn để sự khác biệt giữa V2, V3 và V5 thể hiện rõ.


In [ ]:

IMAGE_NAME = "02.png"

image_size = (512, 512)
patch_size = 7
search_window_size = 21
block_size = (16, 16)
h = 0.12

noise_sigma = 0.08
rng_seed = 42

print("Image size:", image_size)
print("Patch:", patch_size, "x", patch_size)
print("Search:", search_window_size, "x", search_window_size)
print("Block:", block_size)
print("h:", h)



## 4. Tải ảnh tự nhiên và thêm Gaussian noise

Ảnh được chuẩn hóa về `float32` trong miền `[0,1]`.


In [ ]:

image_path = ROOT / "data" / "input" / IMAGE_NAME

clean = io.imread(image_path)

if clean.ndim == 3:
    clean = color.rgb2gray(clean)

clean = transform.resize(
    clean,
    image_size,
    anti_aliasing=True,
).astype(np.float32)

if clean.max() > 1.0:
    clean /= 255.0

rng = np.random.default_rng(rng_seed)

noisy = np.clip(
    clean
    + rng.normal(
        0.0,
        noise_sigma,
        clean.shape,
    ).astype(np.float32),
    0.0,
    1.0,
).astype(np.float32)

print("Clean shape:", clean.shape)
print("Clean range:", float(clean.min()), float(clean.max()))
print("Noisy range:", float(noisy.min()), float(noisy.max()))


In [ ]:

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(clean, cmap="gray", vmin=0, vmax=1)
plt.title("Ảnh sạch")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(noisy, cmap="gray", vmin=0, vmax=1)
plt.title("Ảnh nhiễu")
plt.axis("off")

plt.tight_layout()
plt.show()



# 5. Correctness check với CPU baseline

CPU naive rất chậm với `512×512 / 7×7 / 21×21`, vì vậy correctness chỉ cần chạy trên crop nhỏ.

Mục tiêu ở đây là kiểm tra:

\[
CPU \approx V2 \approx V3 \approx V5
\]

V5 là một **exact dense branch**, nên `allclose=True` là tiêu chí quan trọng.


In [ ]:

correctness_input = noisy[:16, :16].copy()

print("Correctness input:", correctness_input.shape)

t0 = time.perf_counter()
out_cpu_small = nlm_cpu_naive(
    correctness_input,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
)
cpu_small_seconds = time.perf_counter() - t0

out_v2_small = nlm_gpu_v2(
    correctness_input,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
)

out_v3_small = nlm_gpu_v3(
    correctness_input,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
)

out_v5_small = nlm_gpu_v5(
    correctness_input,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
)

def diff_stats(reference, candidate):
    diff = np.abs(reference - candidate)
    return {
        "MAE": float(np.mean(diff)),
        "Max error": float(np.max(diff)),
        "allclose": bool(np.allclose(
            reference,
            candidate,
            rtol=1e-4,
            atol=1e-4,
        )),
    }

correctness_rows = []

for name, output in [
    ("CPU vs V2", out_v2_small),
    ("CPU vs V3", out_v3_small),
    ("CPU vs V5", out_v5_small),
]:
    row = {"Comparison": name}
    row.update(diff_stats(out_cpu_small, output))
    correctness_rows.append(row)

correctness_df = pd.DataFrame(correctness_rows)

print("CPU runtime on 16×16 crop:", f"{cpu_small_seconds:.4f} s")
display(correctness_df)



### Diễn giải correctness

Nếu V5 đúng như thiết kế, ta kỳ vọng:

- `MAE` ở mức rất nhỏ, thường khoảng `1e-6` hoặc thấp hơn.
- `Max error` cũng rất nhỏ.
- `allclose = True`.

Nếu V5 khớp CPU/V2/V3, ta có thể khẳng định rằng V5 **không thay đổi thuật toán NLM**, mà chỉ tối ưu implementation.



# 6. Chạy V3 và V5 trên ảnh 512×512

CPU không được chạy full-size vì đây là implementation naive và có thể mất rất lâu.

Ở phần này ta chỉ chạy các GPU version trên workload benchmark chính.


In [ ]:

# Warm-up one call before taking outputs
_ = nlm_gpu_v3(
    noisy,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
)

_ = nlm_gpu_v5(
    noisy,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
)

cp.cuda.Stream.null.synchronize()

out_v3 = nlm_gpu_v3(
    noisy,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
)

out_v5 = nlm_gpu_v5(
    noisy,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
)

print("V3 output:", out_v3.shape, out_v3.dtype)
print("V5 output:", out_v5.shape, out_v5.dtype)



## 7. V3 vs V5 — kiểm tra output toàn ảnh

V3 và V5 đều là dense exact NLM, nên chúng phải cho output gần như giống nhau về số học.


In [ ]:

v3_v5_diff = np.abs(out_v3 - out_v5)

print("V3 vs V5 MAE:", float(v3_v5_diff.mean()))
print("V3 vs V5 Max error:", float(v3_v5_diff.max()))
print(
    "V3 vs V5 allclose:",
    np.allclose(
        out_v3,
        out_v5,
        rtol=1e-4,
        atol=1e-4,
    ),
)



# 8. Chất lượng ảnh — PSNR và SSIM

Vì V3 và V5 thực hiện cùng dense NLM, PSNR/SSIM của chúng phải gần như giống nhau.

Đây không phải là phần V5 dự kiến cải thiện; mục tiêu chính của V5 là **runtime** trong khi giữ nguyên correctness.


In [ ]:

def quality_metrics(reference, test):
    return {
        "PSNR (dB)": peak_signal_noise_ratio(
            reference,
            test,
            data_range=1.0,
        ),
        "SSIM": structural_similarity(
            reference,
            test,
            data_range=1.0,
        ),
    }

quality_rows = []

for name, image in [
    ("Noisy", noisy),
    ("GPU V3", out_v3),
    ("GPU V5", out_v5),
]:
    row = {"Version": name}
    row.update(quality_metrics(clean, image))
    quality_rows.append(row)

quality_df = pd.DataFrame(quality_rows)
display(quality_df)


In [ ]:

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(noisy, cmap="gray", vmin=0, vmax=1)
plt.title("Noisy")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(out_v3, cmap="gray", vmin=0, vmax=1)
plt.title("GPU V3")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(out_v5, cmap="gray", vmin=0, vmax=1)
plt.title("GPU V5")
plt.axis("off")

plt.tight_layout()
plt.show()



# 9. Benchmark End-to-End

**End-to-End** đo toàn bộ thời gian khi gọi API công khai, bao gồm các thành phần như:

- CPU-side preparation / reflect padding,
- host-to-device transfer,
- GPU computation,
- device-to-host transfer.

Ta dùng benchmark helper của project với warm-up để tránh first-call compile/warm-up làm sai kết quả.


In [ ]:

# Pre-warm nhiều lần để ổn định clock / cache / JIT state
for _ in range(15):
    _ = nlm_gpu_v3(
        noisy,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
    )
    _ = nlm_gpu_v5(
        noisy,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
    )

cp.cuda.Stream.null.synchronize()

v3_e2e = benchmark_gpu_end_to_end(
    function=lambda: nlm_gpu_v3(
        noisy,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
    ),
    warmup_runs=10,
    measured_runs=50,
)

v5_e2e = benchmark_gpu_end_to_end(
    function=lambda: nlm_gpu_v5(
        noisy,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
    ),
    warmup_runs=10,
    measured_runs=50,
)

v3_e2e_ms = v3_e2e["mean_seconds"] * 1000.0
v5_e2e_ms = v5_e2e["mean_seconds"] * 1000.0

e2e_speedup = v3_e2e_ms / v5_e2e_ms
e2e_reduction = (1.0 - v5_e2e_ms / v3_e2e_ms) * 100.0

print("V3 E2E mean:", f"{v3_e2e_ms:.4f} ms")
print("V5 E2E mean:", f"{v5_e2e_ms:.4f} ms")
print("V5 speedup vs V3:", f"{e2e_speedup:.4f}x")
print("E2E reduction:", f"{e2e_reduction:.2f}%")



# 10. Benchmark GPU compute-only bằng CUDA Events

Không nên gọi phần này là “single-kernel only” cho V3 vì V3 có thể bao gồm nhiều CUDA kernels trong pipeline.

Tên chính xác hơn là:

> **GPU compute-only**

Ta chuẩn bị buffers trước, sau đó dùng CUDA Events để chỉ đo phần GPU computation.


In [ ]:

import nlm.gpu_v3 as gpu_v3_module

def _call_with_supported_kwargs(fn, **kwargs):
    sig = inspect.signature(fn)
    supported = {
        k: v
        for k, v in kwargs.items()
        if k in sig.parameters
    }
    return fn(**supported)

def _extract_launcher(prepared):
    if callable(prepared):
        return prepared

    if isinstance(prepared, tuple):
        for item in prepared:
            if callable(item):
                return item

    if isinstance(prepared, dict):
        for key in (
            "kernel_launcher",
            "pipeline_launcher",
            "launcher",
        ):
            value = prepared.get(key)
            if callable(value):
                return value

    raise TypeError(
        "Không tìm thấy callable launcher trong kết quả chuẩn bị."
    )

def prepare_v3_compute_launcher(image):
    candidate_names = [
        "prepare_gpu_v3_pipeline_launch",
        "prepare_gpu_v3_kernel_launch",
        "create_gpu_v3_pipeline_launcher",
        "create_gpu_v3_kernel_launcher",
    ]

    kwargs = dict(
        image=image,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
        displacement_batch_size=8,
        batch_size=8,
    )

    errors = []

    for name in candidate_names:
        fn = getattr(gpu_v3_module, name, None)
        if fn is None:
            continue

        try:
            prepared = _call_with_supported_kwargs(
                fn,
                **kwargs,
            )
            launcher = _extract_launcher(prepared)
            print("V3 compute launcher:", name)
            return launcher
        except Exception as exc:
            errors.append((name, repr(exc)))

    raise RuntimeError(
        "Không tìm thấy V3 compute launcher phù hợp.\n"
        f"Đã thử: {errors}"
    )

v3_launcher = prepare_v3_compute_launcher(noisy)

v5_prepared = prepare_gpu_v5_kernel_launch(
    image=noisy,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
)

v5_launcher = v5_prepared["kernel_launcher"]

print("Launchers prepared.")


In [ ]:

# Prewarm compute launchers
for _ in range(15):
    v3_launcher()
    v5_launcher()

cp.cuda.Stream.null.synchronize()

v3_compute = benchmark_cuda_kernel(
    v3_launcher,
    warmup_runs=10,
    measured_runs=50,
)

v5_compute = benchmark_cuda_kernel(
    v5_launcher,
    warmup_runs=10,
    measured_runs=50,
)

v3_compute_ms = v3_compute["mean_ms"]
v5_compute_ms = v5_compute["mean_ms"]

compute_speedup = v3_compute_ms / v5_compute_ms
compute_reduction = (
    1.0 - v5_compute_ms / v3_compute_ms
) * 100.0

print("V3 compute-only:", f"{v3_compute_ms:.4f} ms")
print("V5 compute-only:", f"{v5_compute_ms:.4f} ms")
print("V5 speedup vs V3:", f"{compute_speedup:.4f}x")
print("Compute reduction:", f"{compute_reduction:.2f}%")



## 11. Bảng tổng hợp V3 vs V5

Đây là bảng chính để trả lời câu hỏi nghiên cứu của V5.


In [ ]:

summary_df = pd.DataFrame([
    {
        "Metric": "GPU compute-only (ms)",
        "GPU V3": v3_compute_ms,
        "GPU V5": v5_compute_ms,
        "V5/V3": v5_compute_ms / v3_compute_ms,
        "Speedup": compute_speedup,
    },
    {
        "Metric": "End-to-End (ms)",
        "GPU V3": v3_e2e_ms,
        "GPU V5": v5_e2e_ms,
        "V5/V3": v5_e2e_ms / v3_e2e_ms,
        "Speedup": e2e_speedup,
    },
    {
        "Metric": "PSNR (dB)",
        "GPU V3": quality_metrics(clean, out_v3)["PSNR (dB)"],
        "GPU V5": quality_metrics(clean, out_v5)["PSNR (dB)"],
        "V5/V3": np.nan,
        "Speedup": np.nan,
    },
    {
        "Metric": "SSIM",
        "GPU V3": quality_metrics(clean, out_v3)["SSIM"],
        "GPU V5": quality_metrics(clean, out_v5)["SSIM"],
        "V5/V3": np.nan,
        "Speedup": np.nan,
    },
])

display(summary_df)


In [ ]:

labels = ["Compute-only", "End-to-End"]
v3_values = [v3_compute_ms, v3_e2e_ms]
v5_values = [v5_compute_ms, v5_e2e_ms]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, v3_values, width, label="GPU V3")
plt.bar(x + width/2, v5_values, width, label="GPU V5")

plt.xticks(x, labels)
plt.ylabel("Runtime (ms)")
plt.title("GPU V3 vs GPU V5")
plt.legend()
plt.tight_layout()
plt.show()



# 12. CPU baseline — performance reference trên crop nhỏ

Không nên benchmark CPU naive trên ảnh `512×512` với `7×7 / 21×21`, vì thời gian chạy có thể rất dài.

Thay vào đó, ta dùng crop `16×16` để minh họa khoảng cách giữa CPU naive và GPU implementation trên cùng một input.

**Lưu ý:** kết quả này chỉ là micro-benchmark minh họa, không phải benchmark chính của V5.


In [ ]:

cpu_bench_input = noisy[:16, :16].copy()

def time_once(fn):
    t0 = time.perf_counter()
    result = fn()
    return (time.perf_counter() - t0), result

# warm GPU first
_ = nlm_gpu_v3(
    cpu_bench_input,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
)
_ = nlm_gpu_v5(
    cpu_bench_input,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
)

cp.cuda.Stream.null.synchronize()

cpu_seconds, _ = time_once(
    lambda: nlm_cpu_naive(
        cpu_bench_input,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
    )
)

v3_small_seconds, _ = time_once(
    lambda: nlm_gpu_v3(
        cpu_bench_input,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
    )
)

v5_small_seconds, _ = time_once(
    lambda: nlm_gpu_v5(
        cpu_bench_input,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
    )
)

cpu_compare_df = pd.DataFrame([
    {
        "Version": "CPU naive",
        "Input": "16×16",
        "Time (ms)": cpu_seconds * 1000.0,
    },
    {
        "Version": "GPU V3",
        "Input": "16×16",
        "Time (ms)": v3_small_seconds * 1000.0,
    },
    {
        "Version": "GPU V5",
        "Input": "16×16",
        "Time (ms)": v5_small_seconds * 1000.0,
    },
])

display(cpu_compare_df)

print(
    "CPU / V5 speedup on 16×16:",
    f"{cpu_seconds / v5_small_seconds:.2f}x",
)



# 13. V5 thực sự cải tiến gì so với V2?

### V2

```text
Global Memory
      ↓
Shared Memory tile + halo
      ↓
one thread / output pixel
      ↓
441 candidates
      ↓
generic 7×7 patch distance
```

### V5

```text
Global Memory
      ↓
Shared Memory tile + halo
      ↓
one thread / output pixel
      ↓
cache fixed 7×7 reference patch
      ↓
441 candidates
      ↓
compile-time fixed / unrolled 49-element distance
      ↓
weight + accumulation
```

Điểm quan trọng:

- V5 **không giảm số candidate**.
- V5 **không dùng Candidate Preselection**.
- V5 **không thay đổi công thức distance**.
- V5 **không dùng Product Map của V3**.
- V5 tập trung vào **specialization của direct dense NLM**.

Do đó V5 là một nhánh khác với V3:

\[
\boxed{\text{V3: reformulate computation}}
\]

\[
\boxed{\text{V5: specialize computation}}
\]



# 14. Cách diễn giải kết quả

Có ba trường hợp chính.

### Trường hợp A — V5 nhanh hơn V3

Nếu:

\[
T_{V5} < T_{V3}
\]

thì có thể kết luận rằng trên workload cố định `7×7 / 21×21`, fixed-size specialization của direct NLM đủ mạnh để vượt qua pipeline Cross-Correlation của V3.

### Trường hợp B — V5 gần bằng V3

Nếu runtime chỉ chênh nhẹ, kết luận phù hợp là:

> V5 thu hẹp đáng kể khoảng cách giữa direct NLM và Cross-Correlation bằng compile-time specialization, nhưng chưa tạo lợi thế rõ ràng.

### Trường hợp C — V5 vẫn chậm hơn V3

Điều này **không làm V5 thất bại**.

Nó cho thấy rằng giảm loop/indexing overhead vẫn không bù được lợi ích algorithmic mà V3 đạt được nhờ reformulate patch distance.

Đây vẫn là một kết quả có ý nghĩa:

> **V3 tối ưu ở mức formulation có ảnh hưởng lớn hơn V5 tối ưu ở mức fixed implementation.**



# 15. Kết luận notebook

GPU V5 được xây dựng như một nhánh exact từ GPU V2:

- giữ Shared Memory,
- giữ dense `441` candidates,
- giữ cùng công thức NLM,
- specialize cho `7×7 / 21×21`,
- cache reference patch,
- unroll fixed patch computation.

Correctness được xác nhận bằng CPU baseline, V2 và V3.

Benchmark chính cần báo cáo gồm:

1. **V3 vs V5 GPU compute-only**.
2. **V3 vs V5 End-to-End**.
3. **PSNR / SSIM** để xác nhận chất lượng không thay đổi.
4. **CPU baseline correctness** trên crop nhỏ.
5. CPU micro-benchmark chỉ dùng để minh họa, không dùng làm benchmark full-size.

Kết luận cuối cùng phải dựa trên số đo thực tế, không giả định rằng V5 bắt buộc phải thắng V3.
